# 01 – Data Cleaning

**Goal:** Load the raw Israeli Lotto draw history, standardize it, and keep only draws in the current game format (6 numbers from 1–37 + 1 strong number from 1–7).

**Input:** `data/raw/Lotto.csv` — source: [\[SOURCE URL\]](https://www.pais.co.il/lotto/archive.aspx), downloaded on 2026-10-23.

**Output:** `data/processed/lotto_current_format.csv`

## 1. Load and inspect raw data
The file is encoded in Windows Hebrew (cp1255). Column names are in Hebrew and one trailing column is empty, so I rename the columns and drop the empty one.

In [1]:
import pandas as pd

RAW_PATH = "../data/raw/Lotto.csv"

df = pd.read_csv(RAW_PATH, encoding="cp1255")   # cp1255 = hebrew code in Windows

print(df.shape)
df.head()

(4553, 12)


,הגרלה,תאריך,1,2,3,4,5,6,המספר החזק/נוסף,מספר_זוכים_לוטו,מספר_זוכים_דאבל_לוטו,Unnamed: 11
0,3977,20/10/2026,3,6,24,25,29,30,5,0.0,0.0,NaN
1,3976,17/10/2026,2,7,15,25,29,37,7,0.0,0.0,NaN
2,3975,15/10/2026,18,22,24,25,27,28,5,1.0,0.0,NaN
3,3974,13/10/2026,3,4,18,19,34,35,2,0.0,0.0,NaN
4,3973,10/10/2026,1,4,7,22,32,34,2,0.0,0.0,NaN


In [2]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 4553 entries, 0 to 4552
Data columns (total 12 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   הגרלה                 4553 non-null   int64  
 1   תאריך                 4553 non-null   str    
 2   1                     4553 non-null   int64  
 3   2                     4553 non-null   int64  
 4   3                     4553 non-null   int64  
 5   4                     4553 non-null   int64  
 6   5                     4553 non-null   int64  
 7   6                     4553 non-null   int64  
 8   המספר החזק/נוסף       4553 non-null   int64  
 9   מספר_זוכים_לוטו       2514 non-null   float64
 10  מספר_זוכים_דאבל_לוטו  2137 non-null   float64
 11  Unnamed: 11           0 non-null      float64
dtypes: float64(3), int64(8), str(1)
memory usage: 427.0 KB


In [3]:
df = df.drop(columns=["Unnamed: 11"])   # пустой столбец от лишней запятой в конце строк

df.columns = ["draw_id", "date", "n1", "n2", "n3", "n4", "n5", "n6",
              "strong", "winners_lotto", "winners_double"]

In [4]:
df["date"] = pd.to_datetime(df["date"], dayfirst=True)
df = df.sort_values("date").reset_index(drop=True)

print(df["date"].min(), df["date"].max())

1968-09-03 00:00:00 2026-10-20 00:00:00


**Observations**
- The raw file has 4,553 draws from 1968-09-03 to 2026-10-20.
- The main numbers and the strong number have no missing values. Winner counts exist only for part of the history (older draws have none).
- The date column was read as plain text and had to be converted to datetime.
- Draw IDs are not monotonic across the full history (about 6,800 in 1968 vs. about 3,900 in 2026), so sorting by date is safer than sorting by ID.

## 2. Find the start of the current format
The game rules changed several times over the years. Instead of guessing a cut-off date, I look for the last draw that breaks the current rules (a main number above 37 or a strong number above 7).

In [5]:
num_cols = ["n1", "n2", "n3", "n4", "n5", "n6"]

violates = (df[num_cols].max(axis=1) > 37) | (df["strong"] > 7)

last_violation = df[violates].iloc[-1]   # df отсортирован по дате, значит последний = самый поздний
print(last_violation)

draw_id                          2233
date              2011-03-01 00:00:00
n1                                 13
n2                                 20
n3                                 25
n4                                 28
n5                                 33
n6                                 35
strong                              8
winners_lotto                     0.0
winners_double                    0.0
Name: 2808, dtype: object


**Observations**
- The last rule-breaking draw is #2233 (2011-03-01): its strong number is 8. The main numbers had already been limited to 1–37 earlier, so the final rule change affected only the strong number.
- The current format therefore starts with the next draw.

## 3. Keep the current format and validate the data
I keep only draws after the last rule violation. Before any analysis, I verify that the data is internally consistent: valid ranges, six distinct main numbers per draw, no duplicate dates, consecutive draw IDs and no missing values.

In [6]:
# Keep only draws after the last rule violation
current = df[df["date"] > last_violation["date"]].copy().reset_index(drop=True)

print(len(current), current["date"].min(), current["date"].max())

1744 2011-03-05 00:00:00 2026-10-20 00:00:00


In [7]:
checks = {
    "main numbers in 1..37": current[num_cols].stack().between(1, 37).all(),
    "strong number in 1..7": current["strong"].between(1, 7).all(),
    "6 unique numbers per draw": (current[num_cols].nunique(axis=1) == 6).all(),
    "no duplicate dates": not current["date"].duplicated().any(),
    "draw ids are consecutive": (current["draw_id"].diff().dropna() == 1).all(),
    "no missing numbers": current[num_cols + ["strong"]].notna().all().all(),
}

for name, ok in checks.items():
    print(f"{name}: {ok}")

print(current[["winners_lotto", "winners_double"]].isna().sum())

main numbers in 1..37: True
strong number in 1..7: True
6 unique numbers per draw: True
no duplicate dates: True
draw ids are consecutive: True
no missing numbers: True
winners_lotto     0
winners_double    0
dtype: int64


**Observations**
- 1,744 draws remain, from 2011-03-05 to 2026-10-20.
- All integrity checks passed: main numbers within 1–37, strong number within 1–7, six unique main numbers in every draw, no duplicate dates, consecutive draw IDs, no missing values.
- Winner counts are fully populated for this period.

## 4. Save the cleaned dataset
The cleaned data is saved separately from the raw file, so later notebooks can start from a reliable dataset without repeating the cleaning.

In [8]:
from pathlib import Path

Path("../data/processed").mkdir(parents=True, exist_ok=True)
current.to_csv("../data/processed/lotto_current_format.csv", index=False)